In [37]:
import pandas as pd
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# Load the parquet file
g_z_all = pd.read_parquet('../historical g spread/bond_z.parquet')

# Now display info - it will show all columns
g_z_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432915 entries, 0 to 432914
Data columns (total 43 columns):
 #   Column                     Non-Null Count   Dtype   
---  ------                     --------------   -----   
 0   Security_1                 432915 non-null  object  
 1   Security_2                 432915 non-null  object  
 2   Last_Spread                432915 non-null  float64 
 3   Z_Score                    432915 non-null  float64 
 4   Max                        432915 non-null  float64 
 5   Min                        432915 non-null  float64 
 6   Last_vs_Max                432915 non-null  float64 
 7   Last_vs_Min                432915 non-null  float64 
 8   Percentile                 432915 non-null  float64 
 9   XCCY                       432915 non-null  float64 
 10  Custom_Sector_1            432915 non-null  object  
 11  Custom_Sector_2            432915 non-null  object  
 12  Rating_1                   432915 non-null  object  
 13  Rating_2      

## All Canadian Bonds

In [38]:
# All Canadian Bonds
g_z_cad = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD')
]
# Select columns
g_z_select_columns = g_z_all[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_select_columns = g_z_select_columns.sort_values(by='Z_Score', ascending=False)


# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_select_columns.columns if g_z_select_columns[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_select_columns.head(50).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
0,FTSCN 2.42 07/18/31,IFCCN 1.928 12/16/30,44.0,5.1,44.0,-9.9,0.0,53.9,99.8,Utility,Insurance Senior,5.1-7.1,5.1-7.1,"2,000,000","2,000,000",10.0,nan
5,HEQCN 6.069 11/14/28,IFCCN 1.928 12/16/30,85.3,4.5,85.3,40.7,0.0,44.6,99.8,Asset Backed,Insurance Senior,3.1-4.1,5.1-7.1,nan,"2,000,000",nan,nan
6,HYDONE 4.39 03/01/34,IFCCN 1.928 12/16/30,57.0,4.5,57.0,12.7,0.0,44.3,99.8,Utility,Insurance Senior,7.1-10.1,5.1-7.1,"5,000,000","2,000,000",4.0,nan
7,DE 4.95 06/14/27,HOMEQU 7.108 12/11/26,-111.8,4.4,-111.8,-140.2,0.0,28.4,99.8,Industrials,Mortgage Provider,1-2.1,1-2.1,"5,000,000","3,000,000",5.0,nan
8,ENBGAS 5.46 10/06/28,IFCCN 1.928 12/16/30,30.7,4.4,30.7,-11.6,0.0,42.3,99.8,Utility,Insurance Senior,3.1-4.1,5.1-7.1,"5,000,000","2,000,000",5.0,nan
9,ENBGAS 4.15 08/17/32,IFCCN 1.928 12/16/30,57.6,4.4,57.6,14.4,0.0,43.2,99.8,Utility,Insurance Senior,7.1-10.1,5.1-7.1,"2,000,000","2,000,000",5.0,nan
10,GZMCN 4.67 09/27/32,IFCCN 1.928 12/16/30,59.1,4.3,59.1,17.8,0.0,41.2,99.8,Utility,Insurance Senior,7.1-10.1,5.1-7.1,"3,000,000","2,000,000",1.0,nan
11,FTSCN 2.42 07/18/31,NWRWPT 5.08 06/01/54,-54.7,4.3,-54.7,-72.4,0.0,17.7,99.8,Utility,Oil & Gas,5.1-7.1,>25.1,"2,000,000","5,000,000",10.0,5.0
12,COCAPS 7.005 09/28/26,IFCCN 1.928 12/16/30,94.3,4.3,94.3,52.4,0.0,41.9,99.8,Non Major Senior Unsecured,Insurance Senior,1-2.1,5.1-7.1,"3,000,000","2,000,000",10.0,nan
14,AL 5.4 06/01/28,HOMEQU 7.108 12/11/26,-56.0,4.3,-56.0,-84.0,0.0,27.9,99.8,Non Financial Maple,Mortgage Provider,2.1-3.1,1-2.1,"2,000,000","3,000,000",3.0,nan


## Same Sector

In [39]:
# Same Sector

# Filter for rows where Custom_Sector_1 equals Custom_Sector_2
g_z_sector = g_z_select_columns[g_z_select_columns['Custom_Sector_1'] == g_z_select_columns['Custom_Sector_2']].copy()

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_select_columns.columns if g_z_select_columns[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_sector.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
41,FTSCN 2.42 07/18/31,LOWMAT 2.433 05/14/31,12.1,4.1,12.1,-10.7,0.0,22.8,99.8,Utility,Utility,5.1-7.1,5.1-7.1,"2,000,000","3,000,000",10.0,10.0
58,GS 6.484 10/24/29,WFC 5.574 07/25/29,11.1,4.0,11.1,-10.5,0.0,21.6,99.8,Financial Maples,Financial Maples,4.1-5.1,3.1-4.1,nan,nan,nan,nan
197,GWOCN 2.981 07/08/50,IFCCN 1.928 12/16/30,90.2,3.8,90.2,55.7,0.0,34.5,99.8,Insurance Senior,Insurance Senior,10.1-25.1,5.1-7.1,nan,"2,000,000",nan,nan
204,CM 5.05 10/07/27,RY 5.228 06/24/30,-8.2,3.8,-8.2,-17.6,0.0,9.4,99.8,Bail In,Bail In,2.1-3.1,4.1-5.1,"25,000,000","10,000,000",2.0,5.0
209,BPYUCN 4 09/30/26,NVACN 7 7/8 07/23/26,273.3,3.8,273.3,-249.4,0.0,522.7,99.8,HY,HY,1-2.1,1-2.1,"2,000,000","2,000,000",73.8,nan
235,FFHCN 3.95 03/03/31,IFCCN 1.928 12/16/30,75.8,3.7,76.3,36.8,-0.6,39.0,99.4,Insurance Senior,Insurance Senior,5.1-7.1,5.1-7.1,"5,000,000","2,000,000",5.0,nan
252,ACACN 4 5/8 08/15/29,NVACN 7 7/8 07/23/26,258.1,3.7,258.1,-140.5,0.0,398.6,99.8,HY,HY,4.1-5.1,1-2.1,"2,000,000","2,000,000",17.0,nan
288,BAC 4.827 07/22/26,HSBC 7.336 11/03/26,45.9,3.7,45.9,-80.3,0.0,126.2,99.8,Financial Maples,Financial Maples,1-2.1,1-2.1,nan,nan,nan,nan
297,BMW 4.41 02/10/27,VW 2.45 12/10/26,0.4,3.7,0.4,-18.6,0.0,19.0,99.8,Auto,Auto,1-2.1,1-2.1,"5,000,000","5,000,000",2.0,-2.0
328,CM 4.9 04/02/27,TD 1.896 09/11/28,23.3,3.7,23.3,-0.9,0.0,24.2,99.8,Bail In,Bail In,1-2.1,3.1-4.1,"20,000,000","5,000,000",4.0,10.0


## Same Term

In [40]:
# Same Term

# Filter for rows where Yrs_Mat_Bucket_1 equals Yrs_Mat_Bucket_2
g_z_term = g_z_select_columns[g_z_select_columns['Yrs_Mat_Bucket_1'] == g_z_select_columns['Yrs_Mat_Bucket_2']].copy()

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_select_columns.columns if g_z_select_columns[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_term.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
0,FTSCN 2.42 07/18/31,IFCCN 1.928 12/16/30,44.0,5.1,44.0,-9.9,0.0,53.9,99.8,Utility,Insurance Senior,5.1-7.1,5.1-7.1,"2,000,000","2,000,000",10.0,nan
7,DE 4.95 06/14/27,HOMEQU 7.108 12/11/26,-111.8,4.4,-111.8,-140.2,0.0,28.4,99.8,Industrials,Mortgage Provider,1-2.1,1-2.1,"5,000,000","3,000,000",5.0,nan
18,ETRHWY 4.45 08/14/31,IFCCN 1.928 12/16/30,68.4,4.2,68.4,24.0,0.0,44.5,99.8,Infastructure,Insurance Senior,5.1-7.1,5.1-7.1,"5,000,000","2,000,000",7.0,nan
21,FTSCN 2.42 07/18/31,TCN 2.05 10/07/30,-6.4,4.2,-6.4,-35.9,0.0,29.4,99.8,Utility,Cable/Telco,5.1-7.1,5.1-7.1,"2,000,000","5,000,000",10.0,nan
31,HYDONE 1.69 01/16/31,IFCCN 1.928 12/16/30,18.4,4.2,18.4,-23.8,0.0,42.2,99.8,Utility,Insurance Senior,5.1-7.1,5.1-7.1,nan,"2,000,000",nan,nan
41,FTSCN 2.42 07/18/31,LOWMAT 2.433 05/14/31,12.1,4.1,12.1,-10.7,0.0,22.8,99.8,Utility,Utility,5.1-7.1,5.1-7.1,"2,000,000","3,000,000",10.0,10.0
50,AL 5.4 06/01/28,HOMEQU 6.552 10/18/27,-79.7,4.1,-79.5,-115.1,-0.2,35.4,99.4,Non Financial Maple,Mortgage Provider,2.1-3.1,2.1-3.1,"2,000,000","3,000,000",3.0,nan
77,HTHROW 3.661 01/13/31,IFCCN 1.928 12/16/30,78.5,4.0,78.5,38.4,0.0,40.1,99.8,Non Financial Maple,Insurance Senior,5.1-7.1,5.1-7.1,"2,000,000","2,000,000",5.0,nan
92,GZMCN 3.04 02/09/32,IFCCN 1.928 12/16/30,55.7,4.0,55.7,14.2,0.0,41.5,99.8,Utility,Insurance Senior,5.1-7.1,5.1-7.1,"2,000,000","2,000,000",4.0,nan
123,FORTFD 4.419 12/23/27,HOMEQU 6.552 10/18/27,-119.3,3.9,-119.3,-153.0,0.0,33.7,99.8,Asset Backed,Mortgage Provider,2.1-3.1,2.1-3.1,"2,000,000","3,000,000",5.0,nan


## Same Ticker

In [41]:
# Same Ticker

# Filter for CAD currency and where Ticker_1 equals Ticker_2
g_z_ticker = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD') &
    (g_z_all['Ticker_1'] == g_z_all['Ticker_2'])
].copy()

# Select the same columns as before
g_z_ticker_select = g_z_ticker[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_ticker_select = g_z_ticker_select.sort_values(by='Z_Score', ascending=False)

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_ticker_select.columns if g_z_ticker_select[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_ticker_select.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
509,RY 4.612 07/26/27,RY 5.228 06/24/30,-14.9,3.4,-14.9,-25.2,0.0,10.3,99.8,Bail In,Bail In,1-2.1,4.1-5.1,"5,000,000","10,000,000",3.0,5.0
628,ONT 2 1/4 12/02/31,ONT 3.65 06/02/33,-11.6,3.4,-11.6,-15.9,0.0,4.3,99.8,Provincials,Provincials,5.1-7.1,7.1-10.1,nan,nan,nan,nan
1438,FTSCN 2.42 07/18/31,FTSCN 4.897 05/27/54,-46.0,3.1,-44.2,-61.6,-1.8,15.5,99.0,Utility,Utility,5.1-7.1,>25.1,"2,000,000","5,000,000",10.0,3.0
1600,FTSCN 2.42 07/18/31,FTSCN 2.632 06/08/51,-44.5,3.1,-42.6,-59.2,-2.0,14.7,99.4,Utility,Utility,5.1-7.1,>25.1,"2,000,000","5,000,000",10.0,nan
1661,FTSCN 2.42 07/18/31,FTSCN 4.862 05/26/53,-45.2,3.1,-43.6,-60.7,-1.7,15.5,99.0,Utility,Utility,5.1-7.1,>25.1,"2,000,000","5,000,000",10.0,3.0
1801,RY 4.632 05/01/28,RY 5.228 06/24/30,-8.3,3.0,-8.3,-20.5,0.0,12.2,99.8,Bail In,Bail In,2.1-3.1,4.1-5.1,"10,000,000","10,000,000",3.0,5.0
2199,ALACN 5 1/4 01/11/2082,ALACN 8.9 11/10/2083,30.9,2.9,32.9,-27.2,-2.0,58.1,99.0,Non Financial Hybrids,Non Financial Hybrids,>25.1,>25.1,"2,000,000","2,000,000",15.0,16.0
2744,FTSCN 2.42 07/18/31,FTSCN 4.618 05/30/52,-45.9,2.9,-43.7,-60.8,-2.2,14.9,99.4,Utility,Utility,5.1-7.1,>25.1,"2,000,000","5,000,000",10.0,5.0
2839,RY 4.642 01/17/28,RY 5.228 06/24/30,-13.4,2.8,-13.4,-20.2,0.0,6.9,99.8,Bail In,Bail In,2.1-3.1,4.1-5.1,"5,000,000","10,000,000",3.0,5.0
3791,ALACN 5.141 03/14/34,ALACN 8.9 11/10/2083,-103.5,2.7,-103.5,-198.5,0.0,95.1,99.8,Midstream,Non Financial Hybrids,7.1-10.1,>25.1,"2,000,000","2,000,000",5.0,16.0


# What's Rich vs A Specific Bond

In [42]:
# Filter for CAD currency and where Security_1 equals "FTSCN 2.42 07/18/31"
g_z_bondlevel = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD') &
    (g_z_all['Security_1'] == 'FTSCN 2.42 07/18/31')   # change this to flip rich or cheap 
].copy()

# Select the same columns as before
g_z_bondlevel = g_z_ftscn[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_bondlevel = g_z_bondlevel.sort_values(by='Z_Score', ascending=False)

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_bondlevel.columns if g_z_bondlevel[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_bondlevel.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
0,FTSCN 2.42 07/18/31,IFCCN 1.928 12/16/30,44.0,5.1,44.0,-9.9,0.0,53.9,99.8,Utility,Insurance Senior,5.1-7.1,5.1-7.1,"2,000,000","2,000,000",10.0,nan
11,FTSCN 2.42 07/18/31,NWRWPT 5.08 06/01/54,-54.7,4.3,-54.7,-72.4,0.0,17.7,99.8,Utility,Oil & Gas,5.1-7.1,>25.1,"2,000,000","5,000,000",10.0,5.0
19,FTSCN 2.42 07/18/31,TD 1.896 09/11/28,39.3,4.2,39.3,13.9,0.0,25.4,99.8,Utility,Bail In,5.1-7.1,3.1-4.1,"2,000,000","5,000,000",10.0,10.0
21,FTSCN 2.42 07/18/31,TCN 2.05 10/07/30,-6.4,4.2,-6.4,-35.9,0.0,29.4,99.8,Utility,Cable/Telco,5.1-7.1,5.1-7.1,"2,000,000","5,000,000",10.0,nan
41,FTSCN 2.42 07/18/31,LOWMAT 2.433 05/14/31,12.1,4.1,12.1,-10.7,0.0,22.8,99.8,Utility,Utility,5.1-7.1,5.1-7.1,"2,000,000","3,000,000",10.0,10.0
44,FTSCN 2.42 07/18/31,TD 1.888 03/08/28,38.2,4.1,38.2,7.7,0.0,30.5,99.8,Utility,Bail In,5.1-7.1,2.1-3.1,"2,000,000","5,000,000",10.0,10.0
49,FTSCN 2.42 07/18/31,LCN 5.336 09/13/52,-72.6,4.1,-72.6,-98.7,0.0,26.0,99.8,Utility,Retail/Grocers,5.1-7.1,>25.1,"2,000,000","5,000,000",10.0,6.0
52,FTSCN 2.42 07/18/31,IGMCN 5.426 05/26/53,-57.0,4.1,-57.0,-82.9,0.0,25.9,99.8,Utility,Insurance Senior,5.1-7.1,>25.1,"2,000,000","5,000,000",10.0,nan
132,FTSCN 2.42 07/18/31,GWOCN 2.981 07/08/50,-46.1,3.9,-46.1,-65.6,0.0,19.4,99.8,Utility,Insurance Senior,5.1-7.1,10.1-25.1,"2,000,000","5,000,000",10.0,nan
140,FTSCN 2.42 07/18/31,NWRWPT 3 3/4 06/01/51,-54.5,3.9,-54.5,-74.6,0.0,20.0,99.8,Utility,Oil & Gas,5.1-7.1,>25.1,"2,000,000","5,000,000",10.0,5.0


## VS What We Own

In [ ]:
# Filter for CAD currency and where Own? equals 1
g_z_own = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD') &
    (g_z_all['Own?'] == 1)
].copy()

# Select the same columns as before
g_z_own_select = g_z_own[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_own_select = g_z_own_select.sort_values(by='Z_Score', ascending=False)

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_own_select.columns if g_z_own_select[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_own_select.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
7,DE 4.95 06/14/27,HOMEQU 7.108 12/11/26,-111.8,4.4,-111.8,-140.2,0.0,28.4,99.8,Industrials,Mortgage Provider,1-2.1,1-2.1,"5,000,000","3,000,000",5.0,nan
14,AL 5.4 06/01/28,HOMEQU 7.108 12/11/26,-56.0,4.3,-56.0,-84.0,0.0,27.9,99.8,Non Financial Maple,Mortgage Provider,2.1-3.1,1-2.1,"2,000,000","3,000,000",3.0,nan
28,DE 4.63 04/04/29,HOMEQU 7.108 12/11/26,-93.7,4.2,-93.7,-120.5,0.0,26.8,99.8,Industrials,Mortgage Provider,3.1-4.1,1-2.1,"5,000,000","3,000,000",5.0,nan
38,DE 5.17 09/15/28,HOMEQU 7.108 12/11/26,-98.2,4.1,-98.2,-124.3,0.0,26.1,99.8,Industrials,Mortgage Provider,3.1-4.1,1-2.1,"5,000,000","3,000,000",4.0,nan
50,AL 5.4 06/01/28,HOMEQU 6.552 10/18/27,-79.7,4.1,-79.5,-115.1,-0.2,35.4,99.4,Non Financial Maple,Mortgage Provider,2.1-3.1,2.1-3.1,"2,000,000","3,000,000",3.0,nan
56,EAGCCT 4.916 06/17/29,HOMEQU 7.108 12/11/26,-81.2,4.1,-81.2,-109.5,0.0,28.3,99.8,Asset Backed,Mortgage Provider,3.1-4.1,1-2.1,"2,000,000","3,000,000",5.0,nan
59,EAGCCT 4.916 06/17/29,HOMEQU 6.552 10/18/27,-104.9,4.0,-104.9,-134.7,0.0,29.8,99.8,Asset Backed,Mortgage Provider,3.1-4.1,2.1-3.1,"2,000,000","3,000,000",5.0,nan
89,DE 4.95 06/14/27,HOMEQU 6.552 10/18/27,-135.5,4.0,-135.5,-164.1,0.0,28.6,99.8,Industrials,Mortgage Provider,1-2.1,2.1-3.1,"5,000,000","3,000,000",5.0,nan
123,FORTFD 4.419 12/23/27,HOMEQU 6.552 10/18/27,-119.3,3.9,-119.3,-153.0,0.0,33.7,99.8,Asset Backed,Mortgage Provider,2.1-3.1,2.1-3.1,"2,000,000","3,000,000",5.0,nan
134,FORTFD 4.419 12/23/27,HOMEQU 7.108 12/11/26,-95.6,3.9,-95.6,-125.0,0.0,29.4,99.8,Asset Backed,Mortgage Provider,2.1-3.1,1-2.1,"2,000,000","3,000,000",5.0,nan


## VS What We Own Filtered For What We Can Trade

In [46]:
# Filter for CAD currency, owned securities, and additional size/bid-offer filters
g_z_own_executable = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD') &
    (g_z_all['Own?'] == 1) &
    (g_z_all['Size @ Best Offer_runs1'] > 2000000) &
    (g_z_all['Size @ Best Bid_runs2'] > 2000000) &
    (g_z_all['Bid/Offer_runs1'] < 3) &
    (g_z_all['Bid/Offer_runs2'] < 3)
].copy()

# Select the same columns as before
g_z_own_executable_select = g_z_own_executable[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_own_executable_select = g_z_own_executable_select.sort_values(by='Z_Score', ascending=False)

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_own_executable_select.columns if g_z_own_executable_select[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_own_executable_select.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
3069,CNHI 4.8 03/25/27,MBGGR 5.12 06/27/28,0.6,2.8,0.6,-25.5,0.0,26.1,99.8,Industrials,Auto,1-2.1,2.1-3.1,"4,000,000","3,000,000",2.0,1.0
5742,CNHI 4.8 03/25/27,OMERS 4.96 02/10/31,-7.6,2.5,-7.6,-23.4,0.0,15.8,99.8,Industrials,Pension,1-2.1,5.1-7.1,"4,000,000","3,000,000",2.0,2.0
9558,COAGAS 4.691 09/30/29,MBGGR 5.12 06/27/28,0.4,2.3,0.4,-29.2,0.0,29.6,99.8,Pipeline,Auto,4.1-5.1,2.1-3.1,"5,000,000","3,000,000",2.0,1.0
12760,CM 5.05 10/07/27,MBGGR 5.12 06/27/28,-10.1,2.2,-10.1,-41.8,0.0,31.7,99.8,Bail In,Auto,2.1-3.1,2.1-3.1,"25,000,000","3,000,000",2.0,1.0
14630,HNDA 4.873 09/23/27,MBGGR 5.12 06/27/28,-7.6,2.1,-7.6,-22.5,-0.0,14.9,99.0,Auto,Auto,2.1-3.1,2.1-3.1,"5,000,000","3,000,000",2.0,1.0
23940,BMW 4.41 02/10/27,MBGGR 5.12 06/27/28,-15.1,1.9,-14.0,-33.3,-1.2,18.2,98.6,Auto,Auto,1-2.1,2.1-3.1,"5,000,000","3,000,000",2.0,1.0
26101,CCDJ 4.407 05/19/27,MBGGR 5.12 06/27/28,-14.9,1.9,-14.9,-47.3,0.0,32.4,99.8,Non Major Senior Unsecured,Auto,1-2.1,2.1-3.1,"5,000,000","3,000,000",2.0,1.0
27476,CM 5.05 10/07/27,OMERS 4.96 02/10/31,-18.2,1.9,-16.3,-33.7,-1.9,15.5,96.6,Bail In,Pension,2.1-3.1,5.1-7.1,"25,000,000","3,000,000",2.0,2.0
32219,BMO 3.65 04/01/27,MBGGR 5.12 06/27/28,-16.3,1.8,-16.3,-50.6,0.0,34.3,99.8,Bail In,Auto,1-2.1,2.1-3.1,"10,000,000","3,000,000",2.0,1.0
36237,CM 5.05 10/07/27,TD 5.491 09/08/28,-5.8,1.8,-4.8,-12.1,-1.0,6.3,97.8,Bail In,Bail In,2.1-3.1,3.1-4.1,"25,000,000","20,000,000",2.0,2.0


## VS Decent Bid Offer + Size

In [44]:
# Filter for CAD currency, owned securities, tight bid/offer, and large offer size
g_z_bid_offer = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD') &
    (g_z_all['Bid/Offer_runs1'] < 2) &
    (g_z_all['Size @ Best Offer_runs1'] > 3000000)
].copy()

# Select the same columns as before
g_z_bid_offer_select = g_z_bid_offer[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_bid_offer_select = g_z_bid_offer_select.sort_values(by='Z_Score', ascending=False)

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_bid_offer_select.columns if g_z_bid_offer_select[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_bid_offer_select.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
664,ALACN 2.477 11/30/30,IPLCN 5.091 11/27/51,-103.5,3.3,-103.5,-129.8,0.0,26.2,99.8,Midstream,Midstream,5.1-7.1,>25.1,"5,000,000","5,000,000",-2.0,nan
871,ALACN 2.477 11/30/30,IFCCN 1.928 12/16/30,73.5,3.2,73.5,27.3,0.0,46.2,99.8,Midstream,Insurance Senior,5.1-7.1,5.1-7.1,"5,000,000","2,000,000",-2.0,nan
1280,ALACN 2.477 11/30/30,NVACN 7 7/8 07/23/26,160.8,3.1,160.8,-376.5,0.0,537.3,99.8,Midstream,HY,5.1-7.1,1-2.1,"5,000,000","2,000,000",-2.0,nan
1627,LOWMAT 4.854 10/31/33,NVACN 7 7/8 07/23/26,143.2,3.1,143.2,-403.3,0.0,546.5,99.8,Utility,HY,7.1-10.1,1-2.1,"5,000,000","2,000,000",0.0,nan
2459,CM 1.96 04/21/31,NVACN 7 7/8 07/23/26,97.9,2.9,97.9,-433.2,0.0,531.1,99.8,NVCC Sub Debt,HY,5.1-7.1,1-2.1,"5,000,000","2,000,000",1.0,nan
2544,CM 2.01 07/21/30,NVACN 7 7/8 07/23/26,89.5,2.9,89.5,-459.5,0.0,549.0,99.8,NVCC Sub Debt,HY,4.1-5.1,1-2.1,"5,000,000","2,000,000",1.0,nan
3771,ALACN 2.477 11/30/30,ATRLCN 5.7 03/26/29,7.2,2.7,12.0,-72.7,-4.8,79.8,99.0,Midstream,HY,5.1-7.1,3.1-4.1,"5,000,000",nan,-2.0,nan
6198,ALACN 2.477 11/30/30,ALACN 8.9 11/10/2083,-139.6,2.5,-139.6,-229.0,0.0,89.4,99.8,Midstream,Non Financial Hybrids,5.1-7.1,>25.1,"5,000,000","2,000,000",-2.0,16.0
7640,ALACN 2.477 11/30/30,HNDA 1.646 02/25/28,64.4,2.4,64.4,28.9,0.0,35.5,99.8,Midstream,Auto,5.1-7.1,2.1-3.1,"5,000,000","2,000,000",-2.0,9.0
8154,ALACN 2.477 11/30/30,ALACN 7.35 08/17/2082,-114.3,2.4,-114.3,-226.8,0.0,112.4,99.8,Midstream,Non Financial Hybrids,5.1-7.1,>25.1,"5,000,000","2,000,000",-2.0,24.0


## CAD/USD RV

In [45]:
# Filter for CAD vs USD currency, same sector, same ticker, same maturity bucket
g_z_usd = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'USD') &
    (g_z_all['Custom_Sector_1'] == g_z_all['Custom_Sector_2']) &
    (g_z_all['Ticker_1'] == g_z_all['Ticker_2']) &
    (g_z_all['Yrs_Mat_Bucket_1'] == g_z_all['Yrs_Mat_Bucket_2'])
].copy()

# Select the same columns as before
g_z_usd_select = g_z_usd[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'XCCY',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_usd_select = g_z_usd_select.sort_values(by='Z_Score', ascending=False) #This would be your main toggle

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_usd_select.columns if g_z_usd_select[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_usd_select.head(30).style.format(format_dict))

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,XCCY,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
7158,F 5.581 02/22/27,F 5.8 03/05/27,59.2,2.4,62.0,-37.3,-2.9,96.5,99.0,68.5,Auto,Auto,1-2.1,1-2.1,"5,000,000",nan,3.0,nan
21774,F 5.242 05/23/28,F 6.8 05/12/28,52.6,2.0,54.1,-54.8,-1.5,107.4,96.2,48.4,Auto,Auto,2.1-3.1,2.1-3.1,"3,000,000",nan,6.0,nan
37679,F 5.581 02/22/27,F 5.85 05/17/27,40.4,1.7,44.5,-34.9,-4.1,75.3,95.4,44.7,Auto,Auto,1-2.1,1-2.1,"5,000,000",nan,3.0,nan
42307,F 2.961 09/16/26,F 5.8 03/05/27,16.3,1.7,21.9,-62.5,-5.6,78.8,97.8,19.4,Auto,Auto,1-2.1,1-2.1,"2,000,000",nan,4.0,nan
49495,F 5.441 02/09/29,F 5.8 03/08/29,48.3,1.6,53.1,-1.2,-4.8,49.6,95.8,47.5,Auto,Auto,3.1-4.1,3.1-4.1,"3,000,000",nan,10.0,nan
55444,F 6.382 11/10/28,F 6.798 11/07/28,54.1,1.5,61.9,6.7,-7.9,47.4,91.1,55.7,Auto,Auto,3.1-4.1,3.1-4.1,"3,000,000",nan,9.0,nan
73407,F 5.441 02/09/29,F 6.798 11/07/28,59.3,1.4,67.1,8.8,-7.8,50.5,86.3,60.7,Auto,Auto,3.1-4.1,3.1-4.1,"3,000,000",nan,10.0,nan
105868,GM 3.15 02/08/27,GM 5 04/09/27,15.3,1.2,24.1,-22.9,-8.8,38.2,90.3,17.1,Auto,Auto,1-2.1,1-2.1,"3,000,000",nan,5.0,nan
108968,MS 1.779 08/04/27,MS 5.05 01/28/27,12.3,1.1,19.7,-24.3,-7.4,36.6,87.1,32.3,Financial Maples,Financial Maples,1-2.1,1-2.1,"5,000,000",nan,10.0,nan
135888,ENBCN 5 01/19/2082,ENBCN 7 3/8 01/15/2083,31.5,1.0,67.7,-94.0,-36.2,125.5,83.1,21.2,Non Financial Hybrids,Non Financial Hybrids,>25.1,>25.1,"2,000,000",nan,8.0,nan
